In [82]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, softmax
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

In [83]:
path = "/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/data/carOBD/obdiidata"
time_col = 'ENGINE_RUN_TINE ()'

files = [file for file in os.listdir(path) if file.endswith('.csv')]

df_list = []
for file in os.listdir(path):
    if file.endswith('.csv'):
        df = pd.read_csv(f'{path}/{file}', index_col=False)
        df['drive_id'] = file
        df_list.append(df)


print(f'{len(df_list)} files loaded out of {len([f for f in os.listdir(path) if f.endswith(".csv")])}')


# %%
def remove_zero_variance_columns(df: pd.DataFrame, exclude_cols: list[str] = None) -> pd.DataFrame:
    """
    Compute std of each std-computable column (numeric only)
    """
    if exclude_cols is None:
        exclude_cols = []
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    cols_to_check = [col for col in numeric_cols if col not in exclude_cols]
  
    std_df = df[cols_to_check].std()
    zero_variance_cols = std_df[std_df == 0].index.tolist()
  
    print(f'{len(zero_variance_cols)} columns with zero variance: {zero_variance_cols}')
  
    if len(zero_variance_cols) > 0:
        df = df.drop(columns=zero_variance_cols)
  
    return df


# %%
def mean_fill_missing_timestamps_and_remove_duplicates(df: pd.DataFrame, time_col: str, id_cols: list[str] = None) -> pd.DataFrame:
    """
    Remove duplicate timestamps by averaging all numeric columns for each unique timestamp.
    This preserves the overall statistics while removing duplicate entries.
    
    Note: The time column itself is not averaged (it becomes the group key).
    Only numeric columns are averaged when multiple rows share the same timestamp.
    """
    if id_cols is None:
        id_cols = []
    
    existing_id_cols = [col for col in id_cols if col in df.columns]
    
    group_cols = [time_col] + existing_id_cols
    
    agg_dict = {}
    for col in df.columns:
        if col not in group_cols:
            if pd.api.types.is_numeric_dtype(df[col]):
                agg_dict[col] = 'mean'
            else:
                agg_dict[col] = 'first'
  
    df_clean = df.groupby(group_cols, as_index=False).agg(agg_dict)
  
    return df_clean


# %%
def downsample(df, time_col, source_file_col, downsample_factor=2):
    result_dfs = []
    
    for source_file in df[source_file_col].unique():
        file_df = df[df[source_file_col] == source_file].copy()
        
        if len(file_df) < downsample_factor * 2:
            continue
        
        file_df = file_df.sort_values(time_col).reset_index(drop=True)
        
        # Simple decimation without pre-smoothing
        downsampled = file_df.iloc[::downsample_factor].copy()
        downsampled[time_col] = np.arange(len(downsampled)) * downsample_factor
        
        result_dfs.append(downsampled.reset_index(drop=True))
    
    return pd.concat(result_dfs, ignore_index=True)


# %%
def filter_long_drives(df, id_col='drive_id', min_length=608):
    """Keep only drives long enough for your context window"""
    drive_lengths = df.groupby(id_col).size()
    valid_drives = drive_lengths[drive_lengths >= min_length].index
    
    print(f"Keeping {len(valid_drives)}/{df[id_col].nunique()} drives")
    print(f"Dropped {len(df) - df[df[id_col].isin(valid_drives)].shape[0]} timesteps")
    
    return df[df[id_col].isin(valid_drives)].reset_index(drop=True)


# %%
# Combine all dataframes
data = pd.concat(df_list, ignore_index=True)


# Clean up
print(f"Total samples: {len(data):,}")
print(f"Unique drives: {data['drive_id'].nunique()}")


# remove some useless columns
data = data.drop(columns=['WARM_UPS_SINCE_CODES_CLEARED ()', 'TIME_SINCE_TROUBLE_CODES_CLEARED ()'])


data = mean_fill_missing_timestamps_and_remove_duplicates(data, time_col=time_col, id_cols=["drive_id"])
data = remove_zero_variance_columns(data, exclude_cols=["drive_id"])
data = downsample(
    data,
    time_col=time_col,
    source_file_col='drive_id',
    downsample_factor=1
)
data.drop(columns=["drive_id"], inplace=True)


129 files loaded out of 129
Total samples: 304,299
Unique drives: 129
3 columns with zero variance: ['FUEL_AIR_COMMANDED_EQUIV_RATIO ()', 'TIME_RUN_WITH_MIL_ON ()', 'DISTANCE_TRAVELED_WITH_MIL_ON ()']


In [84]:
train_ratio = 0.7
val_ratio = 0.15

train_size = int(len(data) * train_ratio)
val_size = int(len(data) * val_ratio)

train_data = data[:train_size]
val_data = data[train_size:train_size + val_size]
test_data = data[train_size + val_size:]

# Standardize based on training data
scaler = StandardScaler()
train_data_scaled = scaler.fit_transform(train_data)
val_data_scaled = scaler.transform(val_data)
test_data_scaled = scaler.transform(test_data)

print(f"Train: {train_data_scaled.shape}")
print(f"Val: {val_data_scaled.shape}")
print(f"Test: {test_data_scaled.shape}")

def create_sliding_windows(data, window_size=5):
    """
    Create sliding windows for time series
    X: [num_samples, num_features, window_size] - input windows
    y: [num_samples, num_features] - target values (next timestep)
    """
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size].T)  # [num_features, window_size]
        y.append(data[i+window_size])      # [num_features]
    return np.array(X), np.array(y)

window_size = 5
X_train, y_train = create_sliding_windows(train_data_scaled, window_size)
X_val, y_val = create_sliding_windows(val_data_scaled, window_size)
X_test, y_test = create_sliding_windows(test_data_scaled, window_size)

print(f"X_train shape: {X_train.shape}")  # [samples, num_features, window_size]
print(f"y_train shape: {y_train.shape}")  # [samples, num_features]

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_val_tensor = torch.FloatTensor(X_val)
y_val_tensor = torch.FloatTensor(y_val)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)


train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of test batches: {len(test_loader)}")


Train: (57684, 22)
Val: (12361, 22)
Test: (12362, 22)
X_train shape: (57679, 22, 5)
y_train shape: (57679, 22)
Number of training batches: 902
Number of validation batches: 194
Number of test batches: 194


In [85]:
class GraphAttentionLayer(MessagePassing):
    """
    Graph Attention Layer with sensor embeddings (GDN paper Eq. 5-8)
    """
    def __init__(self, in_dim, out_dim, embed_dim):
        super().__init__(aggr='add')
        
        # Feature transformation
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        
        # Attention
        self.att = nn.Linear(2 * (embed_dim + out_dim), 1, bias=False)
        
        self.embed_dim = embed_dim
        self.out_dim = out_dim
        self.leaky_relu = nn.LeakyReLU(0.2)
    
    def forward(self, x, edge_index, embeddings):
        """
        x: [num_nodes, in_dim]
        edge_index: [2, num_edges]
        embeddings: [num_nodes, embed_dim]
        """
        # Transform features
        x_transformed = self.W(x)  # [num_nodes, out_dim]
        
        # Concatenate with embeddings (Eq. 6)
        x_with_embed = torch.cat([embeddings, x_transformed], dim=1)
        
        # Message passing with attention
        out = self.propagate(edge_index, x=x_with_embed)
        
        return F.relu(out)
    
    def message(self, x_i, x_j, edge_index_i):
        """
        Compute attention coefficients (Eq. 7-8)
        """
        # Concatenate source and target
        x_cat = torch.cat([x_i, x_j], dim=-1)  # [num_edges, 2*(embed_dim+out_dim)]
        
        # Clamp input to prevent extreme values
        x_cat = torch.clamp(x_cat, min=-10.0, max=10.0)
        
        # Compute attention score
        alpha = self.att(x_cat)  # [num_edges, 1]
        alpha = self.leaky_relu(alpha)
        
        # Clamp attention scores before softmax
        alpha = torch.clamp(alpha, min=-10.0, max=10.0)
        
        # Normalize with softmax
        alpha = softmax(alpha, edge_index_i)
        
        # Clamp transformed features
        x_j_transformed = torch.clamp(x_j[:, -self.out_dim:], min=-10.0, max=10.0)
        
        # Return weighted message (only transformed features)
        return alpha * x_j_transformed

In [86]:
class GDN(nn.Module):
    """
    GDN with safer initialization
    """
    def __init__(self, num_features, window_size, hidden_dim=64, embed_dim=64, topk=15):
        super().__init__()
        
        self.num_features = num_features
        self.window_size = window_size
        self.hidden_dim = hidden_dim
        self.embed_dim = embed_dim
        self.topk = min(topk, num_features - 1)
        
        self.embeddings = nn.Parameter(torch.randn(num_features, embed_dim) * 0.01)
        
        self.gat1 = GraphAttentionLayer(window_size, hidden_dim, embed_dim)
        self.gat2 = GraphAttentionLayer(hidden_dim, hidden_dim, embed_dim)
        
        self.fc = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(0.2)
        
        self.register_buffer('edge_index', None)
        self._edge_cache = {}
        
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        """Custom weight initialization"""
        if isinstance(module, nn.Linear):
            # Use smaller gain and ensure weights are not too large
            nn.init.xavier_normal_(module.weight, gain=0.1)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
        elif isinstance(module, nn.Parameter):
            # Ensure embeddings are small
            if module.data.size(0) == self.num_features:
                nn.init.normal_(module.data, mean=0.0, std=0.01)
    
    def learn_graph_structure(self):
        """Learn graph structure from embeddings"""
        # Ensure embeddings are not NaN or Inf
        if torch.isnan(self.embeddings).any() or torch.isinf(self.embeddings).any():
            print("Warning: Embeddings contain NaN/Inf, reinitializing...")
            nn.init.normal_(self.embeddings, mean=0.0, std=0.01)
        
        # Safe normalization: handle zero-norm vectors
        norms = torch.norm(self.embeddings, p=2, dim=1, keepdim=True)
        norms = torch.clamp(norms, min=1e-8)  # Prevent division by zero
        embeddings_norm = self.embeddings / norms
        
        # Check for NaN after normalization
        if torch.isnan(embeddings_norm).any():
            print("Warning: Normalized embeddings contain NaN, using identity")
            embeddings_norm = torch.eye(self.num_features, device=self.embeddings.device) * 0.01
        
        similarity = torch.mm(embeddings_norm, embeddings_norm.t())
        
        # Clamp similarity to valid range
        similarity = torch.clamp(similarity, min=-1.0, max=1.0)
        
        # Check for NaN in similarity
        if torch.isnan(similarity).any():
            print("Warning: Similarity matrix contains NaN, using identity")
            similarity = torch.eye(self.num_features, device=self.embeddings.device)
        
        similarity.fill_diagonal_(-1e9)
        
        _, topk_indices = torch.topk(similarity, self.topk, dim=1)
        
        edge_list = []
        for i in range(self.num_features):
            for j in topk_indices[i]:
                edge_list.append([j.item(), i])
        
        edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
        
        # Validate edge_index
        if edge_index.size(0) != 2 or edge_index.size(1) == 0:
            raise ValueError(f"Invalid edge_index: {edge_index.shape}")
        
        return edge_index
    
    def get_batched_edge_index(self, batch_size, device):
        """Get or create cached batched edge index"""
        cache_key = (batch_size, str(device))
        if cache_key in self._edge_cache:
            return self._edge_cache[cache_key]
        
        offsets = torch.arange(batch_size, device=device) * self.num_features
        offsets = offsets.view(1, batch_size, 1)
        
        edge_index_batched = self.edge_index.unsqueeze(1) + offsets
        edge_index_batched = edge_index_batched.reshape(2, -1)
        
        self._edge_cache[cache_key] = edge_index_batched
        return edge_index_batched
    
    def forward(self, x):
        """
        x: [batch_size, num_features, window_size]
        """
        batch_size = x.size(0)
        device = x.device
        
        if self.edge_index is None:
            self.edge_index = self.learn_graph_structure().to(device)
        
        x_flat = x.reshape(batch_size * self.num_features, self.window_size)
        batch_embeddings = self.embeddings.repeat(batch_size, 1)
        
        edge_index_batch = self.get_batched_edge_index(batch_size, device)
        
        # Graph layers with checks
        h1 = self.gat1(x_flat, edge_index_batch, batch_embeddings)
        h1 = self.dropout(h1)
        
        h2 = self.gat2(h1, edge_index_batch, batch_embeddings)
        
        # Element-wise multiplication with embeddings (only if dimensions match)
        # Use embedding projection if needed
        if batch_embeddings.size(1) >= self.hidden_dim:
            h2 = h2 * batch_embeddings[:, :self.hidden_dim]
        else:
            # If embedding dim is smaller, pad or use a projection
            embed_proj = batch_embeddings[:, :min(self.hidden_dim, batch_embeddings.size(1))]
            if embed_proj.size(1) < self.hidden_dim:
                # Pad with ones if needed
                padding = torch.ones(batch_embeddings.size(0), self.hidden_dim - embed_proj.size(1), 
                                   device=batch_embeddings.device)
                embed_proj = torch.cat([embed_proj, padding], dim=1)
            h2 = h2 * embed_proj
        
        forecast = self.fc(h2).squeeze(-1)
        forecast = forecast.reshape(batch_size, self.num_features)
        
        # Clamp output to prevent extreme values
        forecast = torch.clamp(forecast, min=-10.0, max=10.0)
        
        return forecast


In [87]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device('cuda' if torch.cuda.is_available() else 
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")


num_features = X_train.shape[1]
model = GDN(
    num_features=num_features,
    window_size=window_size,
    hidden_dim=64,
    embed_dim=64,
    topk=num_features-1
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), 
    lr=1e-4, 
    weight_decay=1e-4  # Increase from 5e-4
)

scheduler = ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.5, 
    patience=3, 
)
criterion = nn.MSELoss()  # Forecasting loss (Eq. 10)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

def train_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device, non_blocking=True)  # non_blocking with pin_memory
        batch_y = batch_y.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)  # Faster than zero_grad()
        
        y_pred = model(batch_X)
        loss = criterion(y_pred, batch_y)
        
        # Check for NaN before backward
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"NaN/Inf loss detected! Skipping batch.")
            print(f"  y_pred min/max: [{y_pred.min():.4f}, {y_pred.max():.4f}]")
            print(f"  y_pred has NaN: {torch.isnan(y_pred).any()}")
            print(f"  y_pred has Inf: {torch.isinf(y_pred).any()}")
            continue

        loss.backward()
        
        # Gradient clipping to prevent explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)


def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            
            y_pred = model(batch_X)
            loss = criterion(y_pred, batch_y)
            
            total_loss += loss.item()
    
    return total_loss / len(val_loader)


Using device: mps
Model parameters: 6,401


In [88]:
# Diagnostic: Check for NaN/inf in data and model outputs
print("=== Data Diagnostics ===")
print(f"X_train contains NaN: {np.isnan(X_train).any()}")
print(f"X_train contains Inf: {np.isinf(X_train).any()}")
print(f"y_train contains NaN: {np.isnan(y_train).any()}")
print(f"y_train contains Inf: {np.isinf(y_train).any()}")
print(f"X_train min/max: [{X_train.min():.4f}, {X_train.max():.4f}]")
print(f"y_train min/max: [{y_train.min():.4f}, {y_train.max():.4f}]")

print("\n=== Model Diagnostics ===")
# Test forward pass with a single batch
model.eval()
with torch.no_grad():
    sample_X = X_train_tensor[:1].to(device)
    sample_y = y_train_tensor[:1].to(device)
    
    print(f"Sample input shape: {sample_X.shape}")
    print(f"Sample input min/max: [{sample_X.min():.4f}, {sample_X.max():.4f}]")
    
    try:
        y_pred = model(sample_X)
        print(f"Output shape: {y_pred.shape}")
        print(f"Output min/max: [{y_pred.min():.4f}, {y_pred.max():.4f}]")
        print(f"Output contains NaN: {torch.isnan(y_pred).any()}")
        print(f"Output contains Inf: {torch.isinf(y_pred).any()}")
        
        loss = criterion(y_pred, sample_y)
        print(f"Loss value: {loss.item():.6f}")
        print(f"Loss is NaN: {torch.isnan(loss)}")
    except Exception as e:
        print(f"ERROR in forward pass: {e}")
        import traceback
        traceback.print_exc()

# Check model parameters
print("\n=== Parameter Diagnostics ===")
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in {name}")
    if torch.isinf(param).any():
        print(f"Inf in {name}")
    print(f"{name}: min={param.min().item():.6f}, max={param.max().item():.6f}, mean={param.mean().item():.6f}")

# Check embeddings specifically
print(f"\nEmbeddings shape: {model.embeddings.shape}")
print(f"Embeddings min/max: [{model.embeddings.min():.4f}, {model.embeddings.max():.4f}]")


=== Data Diagnostics ===
X_train contains NaN: False
X_train contains Inf: False
y_train contains NaN: False
y_train contains Inf: False
X_train min/max: [-15.8374, 20.0844]
y_train min/max: [-15.8374, 20.0844]

=== Model Diagnostics ===
Sample input shape: torch.Size([1, 22, 5])
Sample input min/max: [-7.6529, 4.0007]
Output shape: torch.Size([1, 22])
Output min/max: [-0.0000, 0.0000]
Output contains NaN: False
Output contains Inf: False
Loss value: 0.926326
Loss is NaN: False

=== Parameter Diagnostics ===
embeddings: min=-0.031016, max=0.034456, mean=0.000160
gat1.W.weight: min=-0.045155, max=0.052094, mean=-0.001128
gat1.att.weight: min=-0.020328, max=0.028309, mean=-0.000026
gat2.W.weight: min=-0.043164, max=0.052761, mean=-0.000074
gat2.att.weight: min=-0.023948, max=0.020279, mean=-0.000157
fc.weight: min=-0.041241, max=0.036449, mean=0.001601
fc.bias: min=0.000000, max=0.000000, mean=0.000000

Embeddings shape: torch.Size([22, 64])
Embeddings min/max: [-0.0310, 0.0345]


In [ ]:
epochs = 50
best_val_loss = float('inf')
patience = 10
patience_counter = 0

train_losses = []
val_losses = []

print("Training GDN with optimized DataLoader...")
print("="*60)

for epoch in tqdm(range(epochs), desc='Epochs'):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    
    # Validate
    val_loss = validate(model, val_loader, criterion, device)
    
    # Check for NaN
    if np.isnan(train_loss) or np.isnan(val_loss):
        print(f"\nNaN detected at epoch {epoch}!")
        print(f"Train loss: {train_loss}, Val loss: {val_loss}")
        break
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    # Update learning rate based on validation loss
    scheduler.step(val_loss)  # ✅ Correct: after validation, with val_loss
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_gdn_model.pt')
        patience_counter = 0
    else:
        patience_counter += 1
    
    if epoch % 5 == 0:
        tqdm.write(f'Epoch {epoch:03d}: Train Loss={train_loss:.6f}, '
                    f'Val Loss={val_loss:.6f}, Best={best_val_loss:.6f}')
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print("="*60)
print(f"Best validation loss: {best_val_loss:.6f}")


Training GDN with optimized DataLoader...


Epochs:   2%|▏         | 1/50 [00:59<48:13, 59.04s/it]

Epoch 000: Train Loss=0.882198, Val Loss=1.132096, Best=1.132096


Epochs:   6%|▌         | 3/50 [03:21<54:49, 69.99s/it]

In [ ]:
# %%
def compute_anomaly_scores(model, data_loader, device):
    """
    Compute anomaly scores using DataLoader
    """
    model.eval()
    all_errors = []
    
    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            batch_X = batch_X.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            
            y_pred = model(batch_X)
            errors = torch.abs(y_pred - batch_y).cpu().numpy()
            all_errors.append(errors)
    
    all_errors = np.vstack(all_errors)
    
    # Robust normalization per sensor
    median = np.median(all_errors, axis=0)
    q75 = np.percentile(all_errors, 75, axis=0)
    q25 = np.percentile(all_errors, 25, axis=0)
    iqr = q75 - q25
    iqr[iqr == 0] = 1
    
    normalized_errors = (all_errors - median) / iqr
    anomaly_scores = np.max(normalized_errors, axis=1)
    
    return anomaly_scores, normalized_errors, all_errors


# Compute scores
val_scores, val_sensor_scores, val_errors = compute_anomaly_scores(
    model, val_loader, device
)
test_scores, test_sensor_scores, test_errors = compute_anomaly_scores(
    model, test_loader, device
)
